In [1]:
import json
import pandas as pd

# Load
with open("coasters_combined.json", "r", encoding="utf-8") as f:
    coasters = json.load(f)

# Fix empty infoboxes - replace {} with None (null equivalent)
for coaster in coasters:
    if coaster.get('infobox') == {}:
        coaster['infobox'] = None

# Basic DataFrame
df = pd.DataFrame(coasters)[['qid', 'label', 'description', 'wikipedia_url']]

# Extract coordinates (and optionally other claims)
def extract_claims(claims):
    res = {}
    if 'P625' in claims and len(claims['P625']) > 0:
        val = claims['P625'][0]['value']
        res['latitude'] = val['latitude']
        res['longitude'] = val['longitude']
    if 'P131' in claims and len(claims['P131']) > 0:
        res['admin_location'] = claims['P131'][0]['value']['id']
    return pd.Series(res)

claims_df = pd.DataFrame(coasters)[['qid', 'claims']]
claims_expanded = claims_df['claims'].apply(extract_claims)
claims_expanded['qid'] = claims_df['qid']

# Merge claims info
df = pd.merge(df, claims_expanded, on='qid', how='left')

# Flatten infobox (now empty dicts are replaced with None)
# Flatten infobox manually (each infobox is a single dict, not a list of records)
infobox_records = []
for coaster in coasters:
    row = dict(coaster['infobox']) if coaster['infobox'] else {}
    row['qid'] = coaster['qid']
    infobox_records.append(row)

infobox_df = pd.DataFrame(infobox_records)
df = pd.merge(df, infobox_df, on='qid', how='left')

print(f"Total coasters: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df.head())



Total coasters: 3114
Columns: ['qid', 'label', 'description', 'wikipedia_url', 'latitude', 'longitude', 'admin_location', 'Location', 'Park\xa0section', 'Coordinates', 'Status', 'Opening\xa0date', 'Cost', 'Replaced', 'Type', 'Manufacturer', 'Model', 'Lift/launch system', 'Height', 'Drop', 'Length', 'Speed', 'Inversions', 'Duration', 'Capacity', 'G-force', 'Restraint\xa0style', 'Height restriction', 'Trains', 'Slogan', 'Designer', 'Max vertical angle', 'Website', 'Opened', 'Owner', 'Operating season', 'Area', 'Total', 'Roller coasters', 'Water rides', 'Closing\xa0date', 'Soft opening date', 'Opening date', 'Closing date', 'Replaced by', 'Operated by', 'Attendance', 'Track\xa0layout', 'General manager', 'Theme', 'First manufactured', 'Locations', 'Built', 'Architect', 'NRHP\xa0referenceNo.', 'Added to NRHP', 'Designated\xa0NHL', 'Attraction type', 'Vehicles', 'Riders per vehicle', 'Hangul', 'Hanja', 'RR', 'MR', 'Name', 'Park section', 'Sponsor', 'Music', 'Replaced\xa0by', '1st Launch', '

In [2]:
import pandas as pd

# Check the raw data
qids = [c['qid'] for c in coasters]
print(f"Total entries: {len(qids)}, Unique qids: {len(set(qids))}")

dupes = pd.Series(qids).value_counts()
print(dupes[dupes > 1].head(10))

Total entries: 1554, Unique qids: 1361
Q648978       5
Q674404       4
Q1547610      4
Q116820068    4
Q586556       3
Q2656090      3
Q1295262      3
Q2308436      3
Q2327576      3
Q2690649      3
Name: count, dtype: int64


In [3]:
seen = set()
deduped = []
for c in df:
    if c['qid'] not in seen:
        seen.add(c['qid'])
        deduped.append(c)
coasters = deduped

TypeError: string indices must be integers, not 'str'

In [ ]:
print(df['qid'].duplicated().sum())              # base df
print(claims_expanded['qid'].duplicated().sum())  # claims
print(infobox_df['qid'].duplicated().sum())       # infobox

In [ ]:
df.to_csv("coasters_combined.csv", index=False, encoding="utf-8")